# SpaceX Falcon 9 — Interactive Visual Analytics with Folium

Builds an interactive map of all three launch sites with per-launch outcome markers and a MarkerCluster layer.

In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

d = pd.read_csv("data/dataset_part_2.csv")

sites = d.groupby("LaunchSite").agg(
    Latitude=("Latitude", "first"),
    Longitude=("Longitude", "first"),
    Launches=("Class", "count"),
    Successes=("Class", "sum"),
).reset_index()
sites["SuccessRate"] = (sites["Successes"] / sites["Launches"] * 100).round(1)
print(sites)

center = [d["Latitude"].mean(), d["Longitude"].mean()]
m = folium.Map(location=center, zoom_start=4.3, tiles="OpenStreetMap")

# Circles + labels for each launch site
for _, row in sites.iterrows():
    folium.Circle(
        location=[row["Latitude"], row["Longitude"]],
        radius=1000,
        color="#0f3460",
        fill=True,
        fill_opacity=0.5,
    ).add_to(m)
    folium.map.Marker(
        [row["Latitude"], row["Longitude"]],
        icon=folium.DivIcon(html=f"""<div style="font-size:11px;font-weight:bold;color:#16213e;background:white;
        padding:2px 4px;border-radius:3px;border:1px solid #0f3460;">{row['LaunchSite']}<br/>{row['SuccessRate']}% success ({int(row['Successes'])}/{int(row['Launches'])})</div>""")
    ).add_to(m)

# Marker cluster of individual launch outcomes
marker_cluster = MarkerCluster(name="Launch outcomes").add_to(m)
for _, row in d.iterrows():
    color = "green" if row["Class"] == 1 else "red"
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        icon=folium.Icon(color=color, icon="rocket" if row["Class"] == 1 else "remove", prefix="fa"),
        popup=f"{row['LaunchSite']} | {row['Date']} | {'Landed' if row['Class']==1 else 'Did not land'}",
    ).add_to(marker_cluster)

folium.LayerControl().add_to(m)
m.save("folium_map.html")
print("saved folium_map.html")


## Findings

- All three launch sites sit at low latitude near open ocean/coastline, standard for range safety.
- KSC LC-39A and VAFB SLC-4E post higher success rates than CCAFS SLC-40, which has handled the most launches.
- Open `folium_map.html` for the full interactive map (pan/zoom, click-to-inspect markers, clustering).